# NMF Benchmark
Run NMF topic modelling with full evaluation metrics: **NMI**, **Purity**, **Precision/Recall/F1**, **Topic Diversity**, and **Coherence (Cᵥ)**. Includes exports for CSV, Top-Words Bar Chart, and UMAP 3D Scatter Plot.

## 1. Installation
Run this cell once to install all required dependencies. Skip if already installed.

In [11]:
import sys

!{sys.executable} -m pip install \
    notebook \
    ipywidgets \
    numpy \
    matplotlib \
    seaborn \
    tqdm \
    scikit-learn \
    spacy \
    gensim \
    umap-learn \
    pandas \
    plotly

# Download spaCy English model
!{sys.executable} -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.1 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 2. Imports

In [1]:
import csv
import json
import time
import logging
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import umap
import spacy
import warnings

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
from sklearn.metrics import normalized_mutual_info_score
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel

from google.colab import drive

warnings.filterwarnings("ignore", category=DeprecationWarning)

drive.mount('/content/drive')

logging.basicConfig(level=logging.WARNING)
print("Imports OK.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Imports OK.


## 3. Configuration
Set all options here before running the rest of the notebook.

In [2]:
# --- Input ---
INPUT_FILE    = '/content/drive/MyDrive/ColabData/dataset_comsci_math.json'   # Path to JSON dataset file
K             = 2             # Number of topics. Set to an int to override auto-detect

# --- Label thresholds ---
THRESHOLD     = 0.3              # Score threshold for multi-labels (OpenAlex concepts)
ABS_THRESHOLD = 0.1              # Absolute probability threshold for multi-label prediction
REL_THRESHOLD = 0.3              # Relative threshold multiplier vs. dominant topic
TARGET_LEVEL  = 0                # Concept level to use (0, 1, or 2)

# --- Exports (set to None to skip) ---
EXPORT_CSV        = 'nmf_results.csv'         # CSV with per-paper scores
EXPORT_BARCHART   = 'nmf_bar.png'             # Top-words bar chart
EXPORT_SCATTER_3D = 'nmf_scatter_3d.html'     # UMAP 3D scatter plot HTML

## 4. NMF Service
Self-contained class wrapping spaCy preprocessing, NMF training, and all export methods.

In [3]:
class NMFService:
    def __init__(self, n_topics=10):
        self.n_topics = n_topics
        self.nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])
        self._setup_stopwords()

        custom_stopwords = self._setup_stopwords()

        self.vectorizer = TfidfVectorizer(
            stop_words=custom_stopwords,
            tokenizer=self.spacy_tokenizer,
            max_df=0.7,
            min_df=5,
            ngram_range=(1, 2)
        )
        self.nmf_model = NMF(
            n_components=n_topics,
            random_state=42,
            init='nndsvd',
            max_iter=500
        )
        self.feature_names    = None
        self.doc_topic_matrix = None

    def _setup_stopwords(self):
        academic_stopwords = [
            'finding', 'findings', 'illustrate', 'significant', 'provide', 'provides', 'potential', 'associated', 'effective', 'aspect', 'aspects', 'challenge', 'challenges',
            'paper', 'study', 'research', 'result', 'results', 'method', 'methodology',
            'proposed', 'propose', 'approach', 'based', 'using', 'used', 'use', 'to', 'we', 'source',
            'analysis', 'model', 'system', 'data', 'application', 'new', 'development',
            'performance', 'conclusion', 'abstract', 'introduction', 'work', 'time',
            'significant', 'shown', 'show', 'demonstrate', 'experiment', 'experimental',
            'university', 'department', 'author', 'et', 'al', 'figure', 'table',
            'high', 'low', 'large', 'small', 'different', 'various', 'property', 'properties', 'increase', 'effect', 'activity',
            'structure', 'compound', 'condition', 'quality', 'entry', 'contain', 'parameter', 'observe', 'report', 'present', 'evaluate'
        ]
        for word in academic_stopwords:
            self.nlp.vocab[word].is_stop = True

        full_stopwords = list(set(list(ENGLISH_STOP_WORDS) + academic_stopwords))
        return full_stopwords

    def spacy_tokenizer(self, text):
        if not text:
            return []
        doc = self.nlp(text)
        return [
            token.lemma_.lower() for token in doc
            if not token.is_stop
            and not token.is_punct
            and not token.like_num
            and len(token) > 2
        ]

    def fit_transform(self, documents):
        print(f"Training NMF with {self.n_topics} topics...")
        tfidf_matrix = self.vectorizer.fit_transform(documents)
        self.feature_names    = self.vectorizer.get_feature_names_out()
        self.doc_topic_matrix = self.nmf_model.fit_transform(tfidf_matrix)
        return self.doc_topic_matrix

    def get_top_words_list(self, n_top_words=10):
        topics_words = []
        for topic in self.nmf_model.components_:
            top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
            topics_words.append([self.feature_names[i] for i in top_features_ind])
        return topics_words

    def calculate_topic_diversity(self, n_top_words=10):
        topics_words = self.get_top_words_list(n_top_words)
        unique_words = set(w for topic in topics_words for w in topic)
        total_words  = sum(len(t) for t in topics_words)
        return len(unique_words) / total_words if total_words else 0

    def calculate_coherence_score(self, documents):
        print("Calculating NMF Coherence (Cᵥ)...")
        tokenized_docs = [self.spacy_tokenizer(doc) for doc in documents]
        topics_words   = self.get_top_words_list(n_top_words=10)
        dictionary     = Dictionary(tokenized_docs)
        try:
            cm = CoherenceModel(
                topics=topics_words, texts=tokenized_docs,
                dictionary=dictionary, coherence='c_v'
            )
            return cm.get_coherence()
        except Exception as e:
            print(f"Coherence calc error: {e}")
            return 0.0

    def export_top_words_barchart(self, output_path, n_top_words=10):
        print("Generating Top Words Bar Chart...")
        cols = 3
        rows = int(np.ceil(self.n_topics / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
        axes = axes.flatten()

        for t_idx, topic in enumerate(self.nmf_model.components_):
            top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
            top_features     = [self.feature_names[i] for i in top_features_ind]
            weights          = topic[top_features_ind]
            ax = axes[t_idx]
            ax.barh(top_features, weights, align='center', color='skyblue')
            ax.invert_yaxis()
            ax.set_title(f'Topic {t_idx}', fontdict={'fontsize': 14, 'fontweight': 'bold'})
            ax.tick_params(axis='both', which='major', labelsize=10)

        for i in range(self.n_topics, len(axes)):
            fig.delaxes(axes[i])

        plt.tight_layout()
        plt.savefig(output_path, dpi=300)
        plt.close()
        print(f"Bar chart saved → {output_path}")

    def export_document_scatter_3d(self, output_path, cluster_ids, n_top_words_for_legend=5):
        print("Running UMAP for 3D visualization (this may take a moment)...")
        reducer   = umap.UMAP(n_components=3, random_state=42, metric='cosine')
        embedding = reducer.fit_transform(self.doc_topic_matrix)

        cluster_legend_map = {}
        for t_idx in set(cluster_ids):
            topic            = self.nmf_model.components_[t_idx]
            top_features_ind = topic.argsort()[:-n_top_words_for_legend - 1:-1]
            keyword_str      = ", ".join([self.feature_names[i] for i in top_features_ind])
            cluster_legend_map[t_idx] = f"Topic {t_idx}: {keyword_str}"

        df = pd.DataFrame({
            'UMAP_1': embedding[:, 0],
            'UMAP_2': embedding[:, 1],
            'UMAP_3': embedding[:, 2],
            'Cluster Focus': [cluster_legend_map[cid] for cid in cluster_ids]
        })

        fig = px.scatter_3d(
            df, x='UMAP_1', y='UMAP_2', z='UMAP_3',
            color='Cluster Focus',
            title='NMF Document Clusters (UMAP 3D Projection)',
            opacity=0.8,
            color_discrete_sequence=px.colors.qualitative.Alphabet
        )
        fig.update_traces(marker=dict(size=4, line=dict(width=0.5, color='white')))
        fig.update_layout(legend=dict(itemsizing='constant'))
        fig.write_html(output_path)
        print(f"UMAP 3D scatter saved → {output_path}")

print("NMFService defined.")

NMFService defined.


## 5. Load Data

In [4]:
target_key_hard  = f'true_label_l{TARGET_LEVEL}'
target_key_multi = f'multi_labels_l{TARGET_LEVEL}'

documents       = []
papers_data     = []
y_true_dominant = []

print(f"Loading data from: {INPUT_FILE}")
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

for item in data:
    text = item.get('text', '')
    if not text:
        text = f"{item.get('title', '')} {item.get('abstract', '')}"

    true_labels_set = set()
    top_label = item.get(target_key_hard)

    if target_key_multi in item and isinstance(item[target_key_multi], list):
        true_labels_set = set(item[target_key_multi])
    elif 'openalex_concepts' in item:
        valid_concepts = [
            c for c in item['openalex_concepts']
            if c.get('level') == TARGET_LEVEL and c.get('score', 0) >= THRESHOLD
        ]
        true_labels_set.update([c['name'] for c in valid_concepts])
        if valid_concepts and not top_label:
            valid_concepts.sort(key=lambda x: x.get('score', 0), reverse=True)
            top_label = valid_concepts[0]['name']
    else:
        if top_label:
            true_labels_set.add(top_label)

    if true_labels_set and top_label and text.strip():
        documents.append(text)
        y_true_dominant.append(top_label)
        papers_data.append({
            "id":          str(item.get('id', 'N/A')),
            "title":       str(item.get('title', 'Unknown Title')).replace('\n', ' ').replace('\r', ''),
            "true_labels": list(true_labels_set),
            "top_label":   str(top_label)
        })

n_topics = K if K else len(set(y_true_dominant))
print(f"Loaded {len(documents)} documents — using {n_topics} topics.")

Loading data from: /content/drive/MyDrive/ColabData/dataset_comsci_math.json
Loaded 264 documents — using 2 topics.


## 6. Train NMF

In [5]:
start_time = time.time()

nmf_service      = NMFService(n_topics=n_topics)
doc_topic_matrix = nmf_service.fit_transform(documents)

execution_time = time.time() - start_time

# Normalize rows so topic weights sum to 1 (NMF scores are not probabilities by default)
row_sums = doc_topic_matrix.sum(axis=1)
row_sums[row_sums == 0] = 1
norm_doc_topic_matrix = doc_topic_matrix / row_sums[:, np.newaxis]

topics_words_list  = nmf_service.get_top_words_list(n_top_words=10)
topic_keywords_map = {i: ", ".join(words) for i, words in enumerate(topics_words_list)}

print(f"Training complete in {execution_time:.2f}s.")

Training NMF with 2 topics...


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Training complete in 9.07s.


## 7. Compute Metrics

In [6]:
# --- NMI & Purity ---
y_pred_hard_ids = np.argmax(norm_doc_topic_matrix, axis=1)
nmi = normalized_mutual_info_score(y_true_dominant, y_pred_hard_ids)

cluster_to_label_map = {}
for cid in range(n_topics):
    indices = [i for i, x in enumerate(y_pred_hard_ids) if x == cid]
    if indices:
        labels_in_cluster = [y_true_dominant[i] for i in indices]
        cluster_to_label_map[cid] = Counter(labels_in_cluster).most_common(1)[0][0]
    else:
        cluster_to_label_map[cid] = "Unknown"

y_pred_mapped = [cluster_to_label_map.get(cid, "Unknown") for cid in y_pred_hard_ids]
purity = sum(1 for yt, yp in zip(y_true_dominant, y_pred_mapped) if yt == yp) / len(y_true_dominant)

# --- Per-paper Precision / Recall / F1 ---
f1_scores, precision_scores, recall_scores = [], [], []

for i, paper_item in enumerate(papers_data):
    true_labels_set = set(paper_item["true_labels"])
    probs    = norm_doc_topic_matrix[i]
    max_prob = max(probs) if len(probs) > 0 else 0
    pred_labels = set()

    for t_id, prob in enumerate(probs):
        if prob > ABS_THRESHOLD and prob >= (max_prob * REL_THRESHOLD):
            mapped = cluster_to_label_map.get(t_id, "Unknown")
            if mapped != "Unknown":
                pred_labels.add(mapped)

    max_tid = int(np.argmax(probs))
    if not pred_labels:
        pred_labels.add(cluster_to_label_map.get(max_tid, "Unknown"))

    intersection = len(true_labels_set & pred_labels)
    p  = intersection / len(pred_labels)    if pred_labels    else 0
    r  = intersection / len(true_labels_set) if true_labels_set else 0
    f1 = 2 * (p * r) / (p + r)             if (p + r) > 0    else 0

    precision_scores.append(p)
    recall_scores.append(r)
    f1_scores.append(f1)

    paper_item.update({
        "predicted_top_topic_id":   str(max_tid),
        "predicted_top_label":      str(cluster_to_label_map.get(max_tid, "Unknown")),
        "predicted_multi_labels":   list(pred_labels),
        "predicted_topic_keywords": topic_keywords_map.get(max_tid, ""),
        "precision":                p,
        "recall":                   r,
        "f1_score":                 f1,
        "topic_distribution":       [float(prob) for prob in probs]
    })

avg_f1     = np.mean(f1_scores)
avg_p      = np.mean(precision_scores)
avg_r      = np.mean(recall_scores)
diversity  = nmf_service.calculate_topic_diversity()
coherence  = nmf_service.calculate_coherence_score(documents)

print("Metrics computed.")

Calculating NMF Coherence (Cᵥ)...
Metrics computed.


## 8. Results Summary

In [7]:
print("=" * 60)
print(f"NMF BENCHMARK RESULTS (Level {TARGET_LEVEL})")
print("=" * 60)
print(f"{'NMI Score (Single)':<30} | {nmi:.4f}")
print(f"{'Purity (Single)':<30} | {purity:.4f}")
print("-" * 60)
print(f"{'Avg Precision (Multi)':<30} | {avg_p:.4f}")
print(f"{'Avg Recall (Multi)':<30} | {avg_r:.4f}")
print(f"{'Avg F1 Score (Multi)':<30} | {avg_f1:.4f}")
print("-" * 60)
print(f"{'Topic Diversity':<30} | {diversity:.4f}")
print(f"{'Topic Coherence (Cv)':<30} | {coherence:.4f}")
print("-" * 60)
print(f"Execution Time: {execution_time:.2f} seconds")
print("=" * 60)

print(f"\nNMF TOPIC MAPPING (First {min(10, n_topics)} Topics)")
print("-" * 60)
for cid in range(min(10, n_topics)):
    label = cluster_to_label_map.get(cid, "Unknown")
    words = ", ".join(topics_words_list[cid]) if cid < len(topics_words_list) else ""
    print(f"Topic {cid:<2} -> {label:<25} | Keywords: {words}")

NMF BENCHMARK RESULTS (Level 0)
NMI Score (Single)             | 0.5367
Purity (Single)                | 0.9053
------------------------------------------------------------
Avg Precision (Multi)          | 0.9242
Avg Recall (Multi)             | 0.8977
Avg F1 Score (Multi)           | 0.8932
------------------------------------------------------------
Topic Diversity                | 1.0000
Topic Coherence (Cv)           | 0.6769
------------------------------------------------------------
Execution Time: 9.07 seconds

NMF TOPIC MAPPING (First 2 Topics)
------------------------------------------------------------
Topic 0  -> Mathematics               | Keywords: interval, confidence, confidence interval, bootstrap, distribution, bootstrap confidence, coefficient variation, poisson, coefficient, variation
Topic 1  -> Computer science          | Keywords: classification, machine, network, feature, technique, learning, algorithm, test, accuracy, prediction


## 9. Export CSV

In [8]:
if EXPORT_CSV:
    with open(EXPORT_CSV, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
        header = [
            'Paper ID', 'Title', 'True Top Label', 'True Multi-Labels',
            'Predicted Top Topic ID', 'Predicted Top Label', 'Predicted Multi-Labels',
            'Predicted Topic Keywords', 'Precision', 'Recall', 'F1-Score'
        ]
        for t in range(n_topics):
            header.append(f"Topic_{t}_Prob")
        writer.writerow(header)

        for p in papers_data:
            row = [
                p.get('id', ''),
                p.get('title', ''),
                p.get('top_label', ''),
                ", ".join([str(x) for x in p.get('true_labels', [])]),
                p.get('predicted_top_topic_id', ''),
                p.get('predicted_top_label', ''),
                ", ".join([str(x) for x in p.get('predicted_multi_labels', [])]),
                p.get('predicted_topic_keywords', ''),
                f"{p.get('precision', 0):.4f}",
                f"{p.get('recall', 0):.4f}",
                f"{p.get('f1_score', 0):.4f}"
            ]
            row.extend([f"{prob:.4f}" for prob in p.get('topic_distribution', [])])
            writer.writerow(row)

    print(f"CSV exported → {EXPORT_CSV}")
else:
    print("CSV export skipped (EXPORT_CSV is None).")

CSV exported → nmf_results.csv


## 10. Export Top-Words Bar Chart

In [9]:
if EXPORT_BARCHART:
    nmf_service.export_top_words_barchart(EXPORT_BARCHART)
else:
    print("Bar chart export skipped (EXPORT_BARCHART is None).")

Generating Top Words Bar Chart...
Bar chart saved → nmf_bar.png


## 11. Export UMAP 3D Scatter Plot

In [10]:
if EXPORT_SCATTER_3D:
    nmf_service.export_document_scatter_3d(EXPORT_SCATTER_3D, y_pred_hard_ids)
else:
    print("UMAP 3D scatter export skipped (EXPORT_SCATTER_3D is None).")

Running UMAP for 3D visualization (this may take a moment)...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP 3D scatter saved → nmf_scatter_3d.html
